<a href="https://colab.research.google.com/github/NoaTal1996/Backbone-Graph/blob/main/src/Topic_Clustring_and_NER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Instalations and Imports

In [ ]:
try:
    # Running inside Google Colab → use !pip
    import google.colab
    print("Running in Colab → installing packages...")
    !pip install -q bertopic
    !pip install -q tweet-preprocessor
    !pip install -q spacy-entity-linker
    !pip install -q emoji
except ImportError:
    # Not running in Colab → do nothing
    pass

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

import re
import emoji

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

import spacy

# Load Data
Labeled Datasets for Research on Information Operations

In [ ]:
from pathlib import Path

dataset_name = "Labeled_Datasets"
file_name = "Ecuador/Ecuador_part_3.gzip.parquet"

# Google Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    file_path = Path("/content/drive/Shareddrives/KeyBoostAuthor") / dataset_name / file_name
    in_colab = True
    print("Running inside Colab. Loading from Google Drive...")

# BGU Slurm Cluster
except ImportError:
    # working directory is: Backbone-Graph/src
    file_path = Path("../data") / dataset_name / file_name
    in_colab = False
    print("Running on Slurm cluster. Loading local file...")


# Load the parquet file
df = pd.read_parquet(file_path)


In [ ]:
df.head()

## Dataset Statistics

In [ ]:
total_posts = len(df)
hashtag_posts = df[df['hashtags'].notna() & (df['hashtags'].str.len() > 0)]
mention_posts = df[df['account_mentions'].notna() & (df['account_mentions'].str.len() > 0)]
url_posts = df[df['urls'].notna() & (df['urls'].str.len() > 0)]

print(f"{'df shape:':<40} {df.shape}")
print(f"{'df posts with hashtags # shape:':<40} {hashtag_posts.shape}  ({len(hashtag_posts)/total_posts*100:.2f}%)")
print(f"{'df posts with account_mentions @ shape:':<40} {mention_posts.shape}  ({len(mention_posts)/total_posts*100:.2f}%)")
print(f"{'df posts with urls shape:':<40} {url_posts.shape}  ({len(url_posts)/total_posts*100:.2f}%)")


print()
print()

print(f"{'**collum name**':<30} **value type**")
for col in df.columns:
    dtype_str = str(df[col].dtype)  # Convert dtype to string
    # get a sample non-null value if exists
    sample_val = df[col].dropna().iloc[0] if df[col].notna().any() else None
    print(f"{col:<30} | {type(sample_val).__name__}")


print()
print()
print(df['post_language'].value_counts().head())

# Clean Text

In [ ]:
def clean_url(url_match):

    url = url_match.group(0)

    # 1. Remove protocol (http:// or https://)
    url = re.sub(r"https?://", "", url)
    url = re.sub(r"www\.", "", url)

    # 2. Remove query parameters (?x=1)
    url = url.split("?")[0]

    # 3. Split into components by "/"
    parts = url.split("/")

    # Domain (first part)
    domain = parts[0].replace(".", "_")  # twitter.com → twitter_com

    # First section (second part of the URL)
    if len(parts) > 1 and parts[1].strip() != "":
        section = parts[1].replace(".", "_")
        token = f" URL_{domain}_{section} "
    else:
        token = f" URL_{domain} "

    return token


In [ ]:
def clean_text(text: str) -> str:

    # lowercase
    text = str(text).lower()

    # URLs → domian + first section
    text = re.sub(r"(https?://\S+|www\.\S+)", clean_url, text)

    # Hashtags → HASHTAG_xxx
    text = re.sub(r"#(\w+)", r" HASHTAG_\1 ", text)

    # Mentions → MENTION_xxx
    text = re.sub(r"@(\w+)", r" MENTION_\1 ", text)

    # emoji → EMOJI_xxx
    text = emoji.demojize(text, delimiters=(" EMOJI_", " "))

    # Collapse spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
df["clean_post_text"] = df["post_text"].apply(clean_text)

## Remove Engllish Stopwords

In [ ]:
from nltk.corpus import stopwords
import nltk

In [ ]:
nltk.download('stopwords')

EXTRA_STOPWORDS = [
    'he', 'she', 'they', 'you', 'i', 'we', 'im', 'hes', 'shes', 'its', 'rt', 'amp', 're',
    'theyre', 'dont', 'doesnt', 'would', 'could', 'like', 'know', 'say', 'said', 'one',
    'look', 'looks', 'think', 'oh', 'yeah', 'guy', 'gonna', 'thing', 'someone', 'people',
    'get', 'got', 'thats', 'just', 'really', 'even', 'still', 'want', 'need', 'needs',
    'me', 'to', 'you', 'the', 'that', 'be', 'my', 'get', 'in', 'and',
     'rt', 'the', 'https', 'to', 'my', 'it', 'on', 'you', 'of', 'for', 'https', 'http', 'RT'
]
all_stopwords_english = set(stopwords.words('english') + EXTRA_STOPWORDS)

def remove_stopwords(text):
    return " ".join([w for w in text.lower().split() if w not in all_stopwords_english])


In [ ]:
df["clean_post_text"] = df["clean_post_text"].apply(remove_stopwords)

In [ ]:
df[["post_text", "clean_post_text"]].head(20)

# Topic Clustering

## Utils


In [ ]:
def make_topic_stats(df):

  topic_stats = (
      df.groupby("topic")
        .agg(
            Count=("topic", "count"),
            IO_Count=("is_control", lambda x: (~x).sum())   # count where is_control == False
        )
        .reset_index()
        .rename(columns={"topic": "Topic"})
  )
  topic_stats["Legitimate_posts"] = topic_stats["Count"] - topic_stats["IO_Count"]

  return topic_stats

In [ ]:
def plot_topic_stats(topic_stats):
    topics = topic_stats["Topic"]
    io_count = topic_stats["IO_Count"]
    legitimate_count = topic_stats["Legitimate_posts"]

    plt.figure(figsize=(12, 6))

    # 🔴 Bottom: IO posts
    plt.bar(topics, io_count, label="IO posts", color="red")

    # 🔵 Top: Non-IO posts stacked above IO
    plt.bar(topics, legitimate_count, bottom=io_count, label="Non-IO posts")

    plt.xlabel("Topic ID")
    plt.ylabel("Number of Posts")
    plt.title("Post Count per Topic")
    plt.xticks(topics)
    plt.grid(axis="y")
    plt.legend()
    plt.tight_layout()
    plt.show()

## BERTopic **English** Topic Clustering

In [ ]:
# For Testing Only
df = df[df["post_language"] == "en"].head(200).copy()
print(f"english_df shape: {df.shape}")

In [ ]:
def english_BERTopic_modeling(posts_text_lst, min_topic_size=10):

    embedding_model = SentenceTransformer("digio/Twitter4SSE")

    topic_model = BERTopic(
        embedding_model=embedding_model,
        language="english",
        min_topic_size=min_topic_size,
        verbose=True
    )

    topics, probs = topic_model.fit_transform(posts_text_lst)

    return topics, probs, topic_model

In [ ]:
posts_text_lst = df["clean_post_text"].to_list()

topics, probs, BERTopic_model = english_BERTopic_modeling(posts_text_lst)

df["topic"] = topics

BERTopic_model.get_topic_info()

In [ ]:
print("BERTopic")
topic_stats = make_topic_stats(df)
plot_topic_stats(topic_stats)

In [ ]:
#BERTopic_model.visualize_topics(custom_labels=True)

## K-Means Topic Clustering (English)

In [ ]:
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
from sentence_transformers import SentenceTransformer

posts_text_lst = df["clean_post_text"].astype(str).to_list()

# Load Twitter-trained model
model = SentenceTransformer("digio/Twitter4SSE")   # trained on tweets

# Emmbeding
embeddings = model.encode(posts_text_lst, show_progress_bar=True)

# Normalize
embeddings = normalize(embeddings)

# Run KMeans clustering

k = 20 # Think how to choose the right number

kmeans_model = KMeans(n_clusters=k, random_state=42, n_init="auto")
labels = kmeans_model.fit_predict(embeddings)

# Add to df
df["Kmeans_topic"] = labels

In [ ]:
df.head()

# Named Entity Recognition (NER)

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

# Load model ONCE
tokenizer = AutoTokenizer.from_pretrained("TweebankNLP/bertweet-tb2_wnut17-ner")
model = AutoModelForTokenClassification.from_pretrained("TweebankNLP/bertweet-tb2_wnut17-ner")

ner_pipeline = pipeline(
    "token-classification",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)


def extract_ner_entities(posts_lst: list):
    """
    Runs NER in batches on a list of text posts.

    Args:
          posts_lst (list): List of strings (cleaned posts).
    Returns
    -------
    all_entities (list[list[str]]): For each post: a list of extracted entity words.
    """

    ner_outputs = ner_pipeline(posts_lst)
    all_entities = []
    for post_output in ner_outputs:
        words = [ent["word"] for ent in post_output]
        all_entities.append(words)

    return all_entities

In [ ]:
# Apply ONCE, no pandas apply, in order to utilize batches.
df["ner_entities"] = extract_ner_entities(df["clean_post_text"].to_list())

In [ ]:
df.loc[df["ner_entities"].astype(bool), ["clean_post_text", "ner_entities"]].head()

# Save to parquet

In [ ]:
from pathlib import Path

if in_colab:
  file_path = Path(file_path)
  output_path = (file_path.parents[2] / "Labeled_Dataset_with_Topic_NER" / file_path.parent.name / file_path.name)
else:
  # working directory is: Backbone-Graph/src
  output_path = Path("../results") / dataset_name / file_name

output_path.parent.mkdir(parents=True, exist_ok=True) # Creates the parent directory if needed

df.to_parquet(output_path, index=False, compression="gzip")

print(output_path)